In [2]:
# --- KOMÓRKA 0: PODŁĄCZ DYSK GOOGLE ---
import os
from google.colab import drive

# Sprawdź, czy Dysk jest już podłączony
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
    print("Dysk Google podłączony.")
else:
    print("Dysk Google już podłączony.")

Mounted at /content/drive
Dysk Google podłączony.


In [3]:
# --- KOMÓRKA 1: BIBLIOTEKI ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # Dodano import seaborn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error, median_absolute_error
import time

# Ustawienia estetyczne wykresów
plt.style.use('seaborn-v0_8-darkgrid')
print("Biblioteki załadowane pomyślnie.")

Biblioteki załadowane pomyślnie.


In [4]:
# --- KOMÓRKA 2: WCZYTANIE DANYCH ---
# Sprawdzamy, czy mamy ścieżkę z poprzednich komórek, jeśli nie - używamy domyślnej
if 'OUTPUT_FILE' in locals():
    print(f"Wczytuję dane z: {OUTPUT_FILE}")
    df = pd.read_csv(OUTPUT_FILE)
else:
    path = '/content/drive/MyDrive/FF_MOTOR_PROJECT/data/final_data.csv'
    print(f"Używam ścieżki awaryjnej: {path}")
    try:
        df = pd.read_csv(path)
        print("Dane wczytane poprawnie.")
    except FileNotFoundError:
        print("BŁĄD: Nie znaleziono pliku. Uruchom najpierw komórkę konfiguracyjną (tę pierwszą w całym notesie)!")
        df = pd.DataFrame()

Używam ścieżki awaryjnej: /content/drive/MyDrive/FF_MOTOR_PROJECT/data/final_data.csv
Dane wczytane poprawnie.


In [5]:
# --- KOMÓRKA 3: INŻYNIERIA CECH ---
if not df.empty:
    print("Tworzę cechy fizyczne: Interakcja, Straty Cieplne, Lag...")

    # 1. INTERAKCJA (Moc = Moment * Prędkość) - BAZA
    df['inter_fr'] = df['setpoint_fr'] * df['speed_fr']
    df['inter_rl'] = df['setpoint_rl'] * df['speed_rl']
    df['inter_rr'] = df['setpoint_rr'] * df['speed_rr']

    # 2. STRATY CIEPLNE (Setpoint^2) - DODATEK 1
    df['heat_fr'] = df['setpoint_fr'] ** 2
    df['heat_rl'] = df['setpoint_rl'] ** 2
    df['heat_rr'] = df['setpoint_rr'] ** 2

    # 3. LAG (Opóźnienie) - DODATEK 2
    # Suma setpointów przesunięta o 1 krok w tył
    df['total_setpoint'] = df['setpoint_fr'] + df['setpoint_rl'] + df['setpoint_rr']
    df['lag_setpoint'] = df['total_setpoint'].shift(1).fillna(0)

    # 4. PODSTAWA DYNAMIKI (Niezbędne dla szeregów czasowych)
    # Przyspieszenie
    df['acc_fr'] = df['speed_fr'].diff().fillna(0)
    df['acc_rl'] = df['speed_rl'].diff().fillna(0)
    df['acc_rr'] = df['speed_rr'].diff().fillna(0)

    # Wygładzanie (Rolling mean)
    df['roll_speed_fr'] = df['speed_fr'].rolling(window=5, min_periods=1).mean()
    df['roll_speed_rl'] = df['speed_rl'].rolling(window=5, min_periods=1).mean()
    df['roll_speed_rr'] = df['speed_rr'].rolling(window=5, min_periods=1).mean()

    # Usuwamy puste wiersze
    df.dropna(inplace=True)
    print("Cechy dodane. Gotowe do podziału danych.")
else:
    print("Brak danych do przetworzenia.")

Tworzę cechy fizyczne: Interakcja, Straty Cieplne, Lag...
Cechy dodane. Gotowe do podziału danych.


In [6]:
# --- KOMÓRKA 4: PODZIAŁ DANYCH ---
if not df.empty:
    X = df.drop(columns=['power'])
    y = df['power']

    # Podział: 80% nauka, 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Dane gotowe.")
    print(f"Zbiór treningowy: {X_train.shape[0]} próbek")
    print(f"Zbiór testowy:    {X_test.shape[0]} próbek")

Dane gotowe.
Zbiór treningowy: 41441 próbek
Zbiór testowy:    10361 próbek


In [7]:
# --- KOMÓRKA 5: TRENING ---
# Definicja modelu
model = GradientBoostingRegressor(
    n_estimators=300,        # Liczba drzew
    learning_rate=0.1,       # Szybkość uczenia
    max_depth=5,             # Głębokość drzewa
    random_state=42,
    validation_fraction=0.1, # 10% treningu używamy do walidacji w locie
    n_iter_no_change=10      # Zatrzymaj jeśli nie ma poprawy
)

print(f"Rozpoczynam trening modelu Gradient Boosting...")
model.fit(X_train, y_train)
print("Trening zakończony sukcesem!")

Rozpoczynam trening modelu Gradient Boosting...
Trening zakończony sukcesem!


In [ ]:
from functions.evaluation import display_model_results

# Pomiar czasu predykcji
start_time = time.time()
y_pred = model.predict(X_test)
total_time = time.time() - start_time

results_gb = display_model_results(
    model_name          = "Gradient Boosting",
    y_true              = y_test,
    y_pred              = y_pred,
    inference_time      = total_time,
    feature_importances = model.feature_importances_,
    feature_names       = X_train.columns.tolist(),
)